# Skript Aldape Data

In [ ]:
# import all needed R packages
library(ChAMP)
library(ChAMPdata)
library(ggplot2)
library(stringr)
library(ggpubr)
library(RColorBrewer)
library(colorspace)
library(tidyr)
library(tibble)
library(dplyr)
library(IlluminaHumanMethylation450kanno.ilmn12.hg19)
source("custom_functions.R")
print("Laden der Bibliotheken erfolgreich abgeschlossen")

In [ ]:
# set the location of the data directory
# the .idat fluorescence files and the samplesheet.csv have to be located inside the directory and it's subdirectories
getwd()
aldape_dir <- "/home/jovyan/datasets/Aldape"
print("Setzen des Datenverzeichnisses erfolgreich abgeschlossen")

In [ ]:
myLoad <- champ.load(directory = aldape_dir,
                     method="ChAMP", # replaces old loading method using minfi
                     methValue="B", # wether to calculate beta- or M-values
                     autoimpute=TRUE, # if values are missing uses the 3 most similar probes and uses the mean of their values for the missing value 
                     filterDetP=TRUE, # filter single probes, whose methylation signal vs the background signal of the slide is not significant (detPcut)
                     ProbeCutoff=0, # remove all probes with higher missing-value-ratio
                     SampleCutoff=0.1, # remove a full sample if the failed probe ratio (based on p value) is higher
                     detPcut=0.01, # significance p-value cutoff for filterDetP
                     filterBeads=TRUE, # filter out probes if the fraction of samples with a beadcount < 3 is higher than beadCutoff
                     beadCutoff=0.05, # acceptable fraction of samples with a beadcount < 3
                     filterNoCG=TRUE, # wether to remove non-cg probes
                     filterSNPs=TRUE, # wether to remove probes that fallnear a SNP (as defined in Nordlund et al. https://link.springer.com/article/10.1186/s13059-021-02529-2) 
                     population=NULL, # can be assigned to specific population according to www.internationalgenome.org/category/population/
                     filterMultiHit=TRUE, # wether to remove probes that align to multiple genomic locations (also according to Nordlund et al.)
                     filterXY=TRUE, # wether to remove probes on x and y chromosomes
                     force=FALSE, # minfi specific parameter
                     arraytype="EPICv1") # microarray type (can be one of "450K" "EPICv1" or "EPICv2") 


In [ ]:
champ.QC(beta = myLoad$beta, # beta values stored in champ.load output
         pheno=myLoad$pd$Sample_Group, # what samplesheet column is your phenotype (where do u expect the major difference between your samples)?
         resultsDir="/home/jovyan/CHAMP_QCimages/") # the plots will be saved in the directory this notebook file is located in

In [ ]:
# for two of the normalization methods we need special input, which we generate below
targets <- read.metharray.sheet(aldape_dir)
rgset <- read.metharray.exp(targets = targets, recursive = TRUE, force=TRUE)
mset <- preprocessRaw(rgset)

#### Peak-based correction normalization (PBC)

In [ ]:
myNormPBC <- champ.norm(beta = myLoad$beta, arraytype = "EPICv1", method = "PBC")
champ.QC(beta = myNormPBC, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/PBC/")

#### Beta-mixture quantile normalization (BMIQ)

In [ ]:
myNormBMIQ <- champ.norm(beta = myLoad$beta, arraytype = "EPICv1", method = "BMIQ")
champ.QC(beta = myNormBMIQ, pheno=myLoad$pd$Sample_Group, resultsDir="/home/jovyan/CHAMP_QCimages/BMIQ/")

#### Functional Normalization (FunNorm)

In [ ]:
myNormFunNrom <- champ.norm(beta = myLoad$beta, arraytype = "EPICv1", method = "FunctionalNormalization", rgSet = rgset)
champ.QC(beta = myNormFunNrom, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/FunNorm/")

#### Subset-quantiles within microarray normalization (SWAN)

In [ ]:
myNormSWAN <- champ.norm(arraytype = "EPICv1", method = "SWAN", rgSet=rgset,mset = mset,beta = NULL)
colnames(myNormSWAN) <- targets$Sample_Name
pheno <- targets$Sample_Group         
names(pheno) <- targets$Sample_Name    # Echte Namen
pheno <- pheno[colnames(myNormSWAN)]   # Reihenfolge anpassen
champ.QC(beta = myNormSWAN, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/SWAN/")

In [ ]:
# save best looking normalization result
myNorm <- 

### Single Value Decomposition (SVD) and Batch Effect removal

In [ ]:
# 
champ.SVD(beta=myNorm,pd=myLoad$pd)
SVD_custom(beta=myNorm, filename = "SVD_Aldape.pdf")

In [ ]:
# Remove Batch Effect 
myCombat <- champ.runCombat(beta=myNorm,pd=myLoad$pd,batchname=c("Slide"))

### The covariates are confounded --> we need to have a closer look at the data:

In [ ]:
# customized MDS-Plot function
group_cols <- c("Slide", "Sample_Group", "Institute", "Sex")
pdf("Aldape_mds_custom.pdf", width = 8, height = 6)
lapply( group_cols, plot_mds_by, dat = myNorm, samples = targets, numPos = 1000)
dev.off()

#### **Cohort information:**
![Cohort](images/aldape_cohort.png)
![Sentrix_ID / Slide](images/aldape_heatmap_sentrix_cohort.png)


#### **Questions:**
- Can we remove the effect of slide and institute?
- What would happen if we removed institute or slide since it is confounded with the tumor type (sample group)?
- How important is checking on data quality (in all steps of collecting data) for my analysis?


In [ ]:
# Calculate DMPs
DMPs <- champ.DMP(beta = myNorm,pheno=myLoad$pd$Sample_Group, adjPVal = 0.05, arraytype = "EPICv1")

In [ ]:
DMP.GUI(DMP=DMPs[[1]],beta=myNorm,pheno=myLoad$pd$Sample_Group)

In [ ]:
# champ-DMP automatically compares the Sample Groups to each other:
colnames(DMPs)
nrow(DMPs$AML_to_HLRCC)
nrow(DMPs$AML_to_Hybrid)
nrow(DMPs$AML_to_ccRCC)
nrow(DMPs$HLRCC_to_Hybrid)
nrow(DMPs$HLRCC_to_ccRCC)
nrow(DMPs$Hybrid_to_ccRCC)

##### We have more than two groups to compare - the analysis of DMRs must be selected. Here you can look at one comparison between two Sample Groups:

In [ ]:
DMRs <- champ.DMR(beta=myNorm,pheno=myLoad$pd$Sample_Group,method="Bumphunter", arraytype="EPICv1", compare.group = c("...","..."))